# Data Cleaning

This notebook is for table :
- awards
- genre
- score
- movie_awards
- movie_genre

Code by Matheus Albert 

# Setup

In [1]:
# Import library
import pandas as pd

# Load the Excel file 
meta = pd.read_excel("metaClean43Brightspace.xlsx")

# Check the excel columns and data types
print(meta.columns)
meta.info()

Index(['url', 'title', 'studio', 'rating', 'runtime', 'cast', 'director',
       'genre', 'summary', 'awards', 'metascore', 'userscore', 'RelDate'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11364 entries, 0 to 11363
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   url        11364 non-null  object        
 1   title      11364 non-null  object        
 2   studio     11014 non-null  object        
 3   rating     10297 non-null  object        
 4   runtime    11109 non-null  float64       
 5   cast       7662 non-null   object        
 6   director   11350 non-null  object        
 7   genre      11344 non-null  object        
 8   summary    5467 non-null   object        
 9   awards     4387 non-null   object        
 10  metascore  11364 non-null  int64         
 11  userscore  9259 non-null   float64       
 12  RelDate    11364 non-null  datetime64[ns]
dtypes: datetim

## Genre

In [2]:
# Change column needed to string
meta['title'] = meta['title'].astype('string')
meta['genre'] = meta['genre'].astype('string')

In [3]:
# Get genre column, split each genre by comma, remove missing values, and unique genres
# Explode turns each items in the list to a row
unique_genres = (
    meta['genre']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
    .unique()
)

In [4]:
# Sort genres alphabetically
unique_genres = sorted(unique_genres)

# Create genre dataframe
genre = pd.DataFrame({
    'genre': unique_genres
})

In [5]:
# Add genreid
genre.insert(0, 'genreid', range(1, len(genre) + 1))

# Check the dataframe
print(genre)

    genreid        genre
0         1       Action
1         2        Adult
2         3    Adventure
3         4    Animation
4         5    Biography
5         6       Comedy
6         7        Crime
7         8  Documentary
8         9        Drama
9        10       Family
10       11      Fantasy
11       12    Film-Noir
12       13      History
13       14       Horror
14       15        Music
15       16      Musical
16       17      Mystery
17       18         News
18       19   Reality-TV
19       20      Romance
20       21       Sci-Fi
21       22        Short
22       23        Sport
23       24    Talk-Show
24       25     Thriller
25       26          War
26       27      Western


In [6]:
# Export genre dataframe to csv
genre.to_csv('genre.csv', index=False)

## Awards

In [7]:
# Get award column, remove missing values, remove white spaces before and after awards
awards = meta['awards'].dropna().str.strip()
print(awards)


1        #18MostDiscussedMovieof2016 , #1MostSharedMovi...
7        #23MostDiscussedMovieof2008 , #27MostSharedMov...
11                             #73MostDiscussedMovieof2000
16           #97BestMovieof2009 , #18MostSharedMovieof2009
21                                #71MostSharedMovieof2009
                               ...                        
11356                             #46MostSharedMovieof2007
11357                             #58MostSharedMovieof2011
11358    #17MostDiscussedMovieof2001 , #40MostSharedMov...
11359    #87MostDiscussedMovieof2016 , #80MostSharedMov...
11362    #80BestMovieof2016 , #11MostDiscussedMovieof20...
Name: awards, Length: 4387, dtype: object


In [8]:
# Extract rank, category and year
award_data = awards.str.extract(r'#(\d+)(.*?)of(\d{4})')
award_data.columns = ['Rank', 'Category', 'Year']

In [9]:
# Add space between lowercase and uppercase letters
award_data['Category'] = (
    award_data['Category']
    .str.replace(r'(?<=[a-z])(?=[A-Z])', ' ', regex=True)
    .str.strip()
)

In [10]:
# Create final awards dataframe without rank
# Remove duplicate, sort based on category and year, reset index
awards_df = (
    award_data[['Category', 'Year']]
    .drop_duplicates()
    .sort_values(['Category', 'Year'])
    .reset_index(drop=True)
)

# Add awardid
awards_df.insert(0, 'awardsid', range(1, len(awards_df) + 1))
print(awards_df)

    awardsid           Category  Year
0          1         Best Movie  2000
1          2         Best Movie  2001
2          3         Best Movie  2002
3          4         Best Movie  2003
4          5         Best Movie  2004
..       ...                ...   ...
61        62  Most Shared Movie  2017
62        63  Most Shared Movie  2018
63        64  Most Shared Movie  2019
64        65  Most Shared Movie  2020
65        66  Most Shared Movie  2021

[66 rows x 3 columns]


In [11]:
# Export awards dataframe to csv
awards_df.to_csv('awards.csv', index=False)

## Movie Lookup

In [12]:
# Remove white spaces before and after title
meta["title"] = meta["title"].str.strip()


In [13]:
# Sort tile by Release Date and title
meta = meta.sort_values(["RelDate","title"]).reset_index(drop=True)

# Add movieid
meta.insert(0, 'movieid', range(1, len(meta) + 1))

# Take only the needed column for movie lookup
movie_lookup = meta[[
    "movieid",
    "title",
    "awards",
    "cast",
    "director",
    "genre",
    "metascore",
    "userscore"
]].copy()

In [14]:
# Export movie lookup to csv
movie_lookup.to_csv("movie_lookup.csv", index=False)

## Movie Genre

In [15]:
# Get movieid and genre, remove missing data
genre_data = meta[['movieid', 'genre']].dropna()

# Split the genre with comma delimiter
genre_data['genre'] = genre_data['genre'].str.split(',')

# Explode turns each items in the list to a row
genre_data = genre_data.explode('genre')

# Check genre_data dataframe
print(genre_data)

       movieid      genre
0            1    Fantasy
0            1      Music
0            1  Animation
0            1     Family
1            2  Adventure
...        ...        ...
11361    11362   Thriller
11361    11362        War
11362    11363     Sci-Fi
11362    11363      Drama
11363    11364      Drama

[27055 rows x 2 columns]


In [16]:
# Left join genre_data with genre
movie_genre = genre_data.merge(
    genre,
    on=['genre'],
    how='left'
)

# Check movie_genre data types
movie_genre.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 27055 entries, 0 to 27054
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieid  27055 non-null  int64 
 1   genre    27055 non-null  object
 2   genreid  27055 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 845.5+ KB


In [17]:
# Take only movieid and genreid without genre
movie_genre = movie_genre[['movieid','genreid']]

# Remove duplicate, subset is used so only same combination of movieid and genreid will be removed
movie_genre = movie_genre.drop_duplicates(
    subset=['movieid', 'genreid']
)


In [18]:
# Export Movie genre to csv
movie_genre.to_csv("movie_genre.csv",index=False)

## Movie Awards

In [19]:
# Get movieid and awards, remove missing value
awards_data = meta[['movieid', 'awards']].dropna()

# Extract Rank, Category, and Year
awards_data[['Rank', 'Category', 'Year']] = awards_data['awards'].str.extract(
    r'#(\d+)(.*?)of(\d{4})'
)

# Add space between lowercase and uppercase letters
awards_data['Category'] = (
    awards_data['Category']
    .str.replace(r'(?<=[a-z])(?=[A-Z])', ' ', regex=True)
    .str.strip()
)

In [20]:
# Left join awards_data with awards 
movie_awards = awards_data.merge(
    awards_df,
    on=['Category', 'Year'],
    how='left'
)

movie_awards.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4387 entries, 0 to 4386
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movieid   4387 non-null   int64 
 1   awards    4387 non-null   object
 2   Rank      4387 non-null   object
 3   Category  4387 non-null   object
 4   Year      4387 non-null   object
 5   awardsid  4387 non-null   int64 
dtypes: int64(2), object(4)
memory usage: 239.9+ KB


In [21]:
# Take only the column needed
movie_awards = movie_awards[['movieid','awardsid','Rank']]


In [22]:
# Export movie_awards to csv
movie_awards.to_csv('movie_awards.csv',index=False)

## Score

In [23]:
# Remove duplicates and take only needed column
score_data = meta[['movieid', 'userscore', 'metascore']].dropna()

# check score data df to see the data types
score_data.info()



<class 'pandas.core.frame.DataFrame'>
Int64Index: 9259 entries, 0 to 11362
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   movieid    9259 non-null   int64  
 1   userscore  9259 non-null   float64
 2   metascore  9259 non-null   int64  
dtypes: float64(1), int64(2)
memory usage: 289.3 KB


In [24]:
# Score data is float, x 10 to match with metascore and convert it to int
score_data['userscore'] = (score_data['userscore']*10).astype(int)

score_data.info()

# Export score_data to csv
score_data.to_csv("score.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
Int64Index: 9259 entries, 0 to 11362
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   movieid    9259 non-null   int64
 1   userscore  9259 non-null   int32
 2   metascore  9259 non-null   int64
dtypes: int32(1), int64(2)
memory usage: 253.2 KB
